In [10]:
import pandas as pd

customers = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\cleaned\customers_cleaned.csv")
orders = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\cleaned\orders_cleaned.csv")
order_items = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\cleaned\order_items_cleaned.csv")
order_payments = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\cleaned\order_payments_cleaned.csv")
order_reviews = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\cleaned\order_reviews_cleaned.csv")
products = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\cleaned\products_cleaned.csv")
sellers = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\cleaned\sellers_cleaned.csv")
geolocation = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\cleaned\geolocation_cleaned.csv")
category_translation = pd.read_csv(r"D:\HCL_PROJECTS\Cart2 Insight\Data\cleaned\category_translation_cleaned.csv")

dataset_shapes = {
    "customers": customers.shape,
    "orders": orders.shape,
    "order_items": order_items.shape,
    "order_payments": order_payments.shape,
    "order_reviews": order_reviews.shape,
    "products": products.shape,
    "sellers": sellers.shape,
    "geolocation": geolocation.shape,
    "translation": category_translation.shape,
}

print(dataset_shapes)

dataset_shapes = {
    "customers": customers.shape,
    "orders": orders.shape,
    "order_items": order_items.shape,
    "order_payments": order_payments.shape,
    "order_reviews": order_reviews.shape,
    "products": products.shape,
    "sellers": sellers.shape,
    "geolocation": geolocation.shape,
    "translation": category_translation.shape,
}

print(dataset_shapes)


{'customers': (99441, 5), 'orders': (99441, 10), 'order_items': (112650, 9), 'order_payments': (103886, 9), 'order_reviews': (99224, 7), 'products': (32951, 15), 'sellers': (3095, 4), 'geolocation': (738327, 5), 'translation': (71, 2)}
{'customers': (99441, 5), 'orders': (99441, 10), 'order_items': (112650, 9), 'order_payments': (103886, 9), 'order_reviews': (99224, 7), 'products': (32951, 15), 'sellers': (3095, 4), 'geolocation': (738327, 5), 'translation': (71, 2)}


In [11]:
order_value = (
    order_items
    .groupby("order_id")
    .agg(
        product_value=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

order_value["total_order_value"] = (
    order_value["product_value"] +
    order_value["freight_value"]
)

order_value.head()

,order_id,product_value,freight_value,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04


In [13]:
order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_cols:
    orders[col] = pd.to_datetime(
        orders[col],
        format="%d-%m-%Y %H:%M",
        errors="coerce"
    )

print(orders[order_date_cols].dtypes)

order_purchase_timestamp         datetime64[s]
order_approved_at                datetime64[s]
order_delivered_carrier_date     datetime64[s]
order_delivered_customer_date    datetime64[s]
order_estimated_delivery_date    datetime64[s]
dtype: object


In [14]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

orders["delivery_days"] = orders["delivery_days"].round(0)

In [15]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

orders["delivery_delay_days"] = orders["delivery_delay_days"].round(0)

In [16]:
print(
    orders[
        [
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_days",
            "delivery_delay_days"
        ]
    ].head(10)
)

  order_purchase_timestamp order_delivered_customer_date  \
0                      NaT                           NaT   
1                      NaT                           NaT   
2                      NaT                           NaT   
3                      NaT                           NaT   
4                      NaT                           NaT   
5                      NaT                           NaT   
6                      NaT                           NaT   
7                      NaT                           NaT   
8                      NaT                           NaT   
9                      NaT                           NaT   

  order_estimated_delivery_date  delivery_days  delivery_delay_days  
0                           NaT            NaN                  NaN  
1                           NaT            NaN                  NaN  
2                           NaT            NaN                  NaN  
3                           NaT            NaN             

In [17]:
import pandas as pd

orders = pd.read_csv(
    r"D:\HCL_PROJECTS\Cart2 Insight\Data\raw\olist_orders_dataset.csv"
)

order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_cols:
    orders[col] = pd.to_datetime(
        orders[col],
        format="%d-%m-%Y %H:%M",
        errors="coerce"
    )

print("Missing values:")
print(orders[order_date_cols].isna().sum())

print("\nData types:")
print(orders[order_date_cols].dtypes)

Missing values:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Data types:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [18]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders["delivery_days"] = orders["delivery_days"].round(0)

In [19]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

orders["delivery_delay_days"] = orders["delivery_delay_days"].round(0)

In [20]:
print(
    orders[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_days",
            "delivery_delay_days"
        ]
    ].head(10)
)

                           order_id order_purchase_timestamp  \
0  e481f51cbdc54678b7cc49136f2d6af7      2017-10-02 10:56:00   
1  53cdb2fc8bc7dce0b6741e2150273451      2018-07-24 20:41:00   
2  47770eb9100c2d0c44946d9cf07ec65d      2018-08-08 08:38:00   
3  949d5b44dbf5de918fe9c16f97b45f8a      2017-11-18 19:28:00   
4  ad21c59c0840e6cb83a9ceb5573f8159      2018-02-13 21:18:00   
5  a4591c265e18cb1dcee52889e2d8acc3      2017-07-09 21:57:00   
6  136cce7faa42fdb2cefd53fdc79a6098      2017-04-11 12:22:00   
7  6514b8ad8028c9f2cc2374ded245783f      2017-05-16 13:10:00   
8  76c6e866289321a7c93b82b54852dc33      2017-01-23 18:29:00   
9  e69bfb5eb88e0ed6a785585b27e16dbf      2017-07-29 11:55:00   

  order_delivered_customer_date order_estimated_delivery_date  delivery_days  \
0           2017-10-10 21:25:00                    2017-10-18            8.0   
1           2018-08-07 15:27:00                    2018-08-13           14.0   
2           2018-08-17 18:06:00                    2018

In [21]:
orders["delivery_performance"] = "Not Delivered"

orders.loc[
    orders["delivery_delay_days"] < 0,
    "delivery_performance"
] = "Early"

orders.loc[
    orders["delivery_delay_days"] == 0,
    "delivery_performance"
] = "On Time"

orders.loc[
    orders["delivery_delay_days"] > 0,
    "delivery_performance"
] = "Delayed"

In [22]:
print(orders["delivery_performance"].value_counts())

delivery_performance
Early            87370
Delayed           7646
Not Delivered     2965
On Time           1460
Name: count, dtype: int64


In [23]:
print(
    orders["delivery_performance"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

delivery_performance
Early            87.86
Delayed           7.69
Not Delivered     2.98
On Time           1.47
Name: proportion, dtype: float64


In [24]:
customer_order_count = (
    orders
    .groupby("customer_id")
    .agg(
        customer_order_count=("order_id", "nunique")
    )
    .reset_index()
)

customer_order_count.head()

,customer_id,customer_order_count
0,00012a2ce6f8dcda20d059ce98491703,1
1,000161a058600d5901f007fab4c27140,1
2,0001fd6190edaaf884bcaf3d49edf079,1
3,0002414f95344307404f0ace7a26f1d5,1
4,000379cdec625522490c315e70c7a9fb,1


In [25]:
orders_features = orders.merge(
    order_value[["order_id", "total_order_value"]],
    on="order_id",
    how="left"
)

In [26]:
customer_spending = (
    orders_features
    .groupby("customer_id")
    .agg(
        customer_total_spending=("total_order_value", "sum")
    )
    .reset_index()
)

customer_spending.head()

,customer_id,customer_total_spending
0,00012a2ce6f8dcda20d059ce98491703,114.74
1,000161a058600d5901f007fab4c27140,67.41
2,0001fd6190edaaf884bcaf3d49edf079,195.42
3,0002414f95344307404f0ace7a26f1d5,179.35
4,000379cdec625522490c315e70c7a9fb,107.01


In [27]:
customer_aov = (
    orders_features
    .groupby("customer_id")
    .agg(
        average_order_value=("total_order_value", "mean")
    )
    .reset_index()
)

customer_aov.head()

,customer_id,average_order_value
0,00012a2ce6f8dcda20d059ce98491703,114.74
1,000161a058600d5901f007fab4c27140,67.41
2,0001fd6190edaaf884bcaf3d49edf079,195.42
3,0002414f95344307404f0ace7a26f1d5,179.35
4,000379cdec625522490c315e70c7a9fb,107.01


In [28]:
customer_features = (
    orders_features
    .groupby("customer_id")
    .agg(
        customer_order_count=("order_id", "nunique"),
        customer_total_spending=("total_order_value", "sum"),
        average_order_value=("total_order_value", "mean")
    )
    .reset_index()
)

customer_features.head()

,customer_id,customer_order_count,customer_total_spending,average_order_value
0,00012a2ce6f8dcda20d059ce98491703,1,114.74,114.74
1,000161a058600d5901f007fab4c27140,1,67.41,67.41
2,0001fd6190edaaf884bcaf3d49edf079,1,195.42,195.42
3,0002414f95344307404f0ace7a26f1d5,1,179.35,179.35
4,000379cdec625522490c315e70c7a9fb,1,107.01,107.01


In [29]:
seller_revenue = (
    order_items
    .assign(
        item_revenue=
        order_items["price"] +
        order_items["freight_value"]
    )
    .groupby("seller_id")
    .agg(
        seller_revenue=("item_revenue", "sum")
    )
    .reset_index()
)

seller_revenue.head()

,seller_id,seller_revenue
0,0015a82c2db000af6aaaf3ae2ecb0532,2748.06
1,001cca7ae9ae17fb1caed9dfb1094831,33934.17
2,001e6ad469a905060d959994f1b41e4f,267.94
3,002100f778ceb8431b7a1020ff7ab48f,2028.16
4,003554e2dce176b5555353e4f3555ac8,139.38


In [30]:
seller_order_count = (
    order_items
    .groupby("seller_id")
    .agg(
        seller_order_count=("order_id", "nunique")
    )
    .reset_index()
)

seller_order_count.head()

,seller_id,seller_order_count
0,0015a82c2db000af6aaaf3ae2ecb0532,3
1,001cca7ae9ae17fb1caed9dfb1094831,200
2,001e6ad469a905060d959994f1b41e4f,1
3,002100f778ceb8431b7a1020ff7ab48f,51
4,003554e2dce176b5555353e4f3555ac8,1


In [31]:
seller_features = (
    order_items
    .assign(
        item_revenue=
        order_items["price"] +
        order_items["freight_value"]
    )
    .groupby("seller_id")
    .agg(
        seller_revenue=("item_revenue", "sum"),
        seller_order_count=("order_id", "nunique")
    )
    .reset_index()
)

seller_features.head()

,seller_id,seller_revenue,seller_order_count
0,0015a82c2db000af6aaaf3ae2ecb0532,2748.06,3
1,001cca7ae9ae17fb1caed9dfb1094831,33934.17,200
2,001e6ad469a905060d959994f1b41e4f,267.94,1
3,002100f778ceb8431b7a1020ff7ab48f,2028.16,51
4,003554e2dce176b5555353e4f3555ac8,139.38,1


In [32]:
customer_features["repeat_customer"] = (
    customer_features["customer_order_count"] > 1
)

In [33]:
customer_features["customer_type"] = customer_features[
    "customer_order_count"
].apply(
    lambda x: "Repeat Customer" if x > 1 else "One-time Customer"
)

In [34]:
customer_features = (
    orders_features
    .groupby("customer_id")
    .agg(
        customer_order_count=("order_id", "nunique"),
        customer_total_spending=("total_order_value", "sum"),
        average_order_value=("total_order_value", "mean")
    )
    .reset_index()
)

customer_features["repeat_customer"] = (
    customer_features["customer_order_count"] > 1
)

customer_features["customer_type"] = customer_features[
    "repeat_customer"
].map({
    True: "Repeat Customer",
    False: "One-time Customer"
})

customer_features.head()

,customer_id,customer_order_count,customer_total_spending,average_order_value,repeat_customer,customer_type
0,00012a2ce6f8dcda20d059ce98491703,1,114.74,114.74,False,One-time Customer
1,000161a058600d5901f007fab4c27140,1,67.41,67.41,False,One-time Customer
2,0001fd6190edaaf884bcaf3d49edf079,1,195.42,195.42,False,One-time Customer
3,0002414f95344307404f0ace7a26f1d5,1,179.35,179.35,False,One-time Customer
4,000379cdec625522490c315e70c7a9fb,1,107.01,107.01,False,One-time Customer
